# Leyes de potencia: coordenadas logarítmicas y elasticidad

Una aplicación de Alpha Calculus, con ajuste, interpretación y supuestos
visibles.

Este cuaderno es autónomo. Se presupone manejo de logaritmos, derivadas
elementales y Python básico. Al terminar podrás interpretar una potencia
como una recta en log-log y distinguir lo que cambia al transformar datos.

1. Fijar razones adimensionales y transportar una función positiva.
2. Ajustar una recta e interpretar su pendiente como elasticidad.
3. Separar error multiplicativo y aditivo, y sus criterios de ajuste.
4. Distinguir media y mediana al volver de la escala logarítmica.

Ejecuta todas las celdas en orden desde un runtime nuevo. Sólo usamos
la biblioteca estándar: no se necesitan otro cuaderno, el libro, red,
datos privados ni acceso a Drive. Las tablas se exportan a
`resultados/potencias_y_elasticidad/` como CSV.

In [1]:
import csv
import json
import math
import platform
from pathlib import Path

RESULTADOS = Path("resultados") / "potencias_y_elasticidad"
RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Precisión de float: aproximadamente 16 cifras significativas.")

Python: 3.14.3
Precisión de float: aproximadamente 16 cifras significativas.


In [2]:
def log_positivo(valor):
    """Coordenada de una cantidad adimensional, positiva y finita."""
    if not math.isfinite(valor) or valor <= 0:
        raise ValueError("La cantidad debe ser positiva y finita.")
    return math.log(valor)

In [3]:
def tabla(titulo, cabecera, filas):
    """Imprime una tabla pequeña, con las columnas alineadas."""
    datos = [[str(valor) for valor in fila] for fila in [cabecera, *filas]]
    anchos = [max(map(len, columna)) for columna in zip(*datos)]
    print(titulo)
    for indice, fila in enumerate(datos):
        print(" | ".join(
                valor.ljust(ancho)
                for valor, ancho in zip(fila, anchos)
                ))
        if indice == 0:
            print("-+-".join("-" * ancho for ancho in anchos))


def exportar_tabla(nombre, cabecera, filas):
    """Guarda los datos de una tabla como CSV UTF-8."""
    with (RESULTADOS / nombre).open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(cabecera)
        escritor.writerows(filas)

## 1. Aplicación: reconocer una ley de potencia y leer su elasticidad

### Situación, reconocimiento y supuestos

Una respuesta positiva puede seguir una ley de escala. Si multiplicar
la entrada por $q$ multiplica la respuesta por $q^b$, una potencia es
una candidata. Un aspecto recto en log-log ayuda a explorar; no demuestra
el modelo ni identifica su ruido.

Fijamos referencias positivas $x_0,y_0$ con sus unidades y trabajamos
con razones adimensionales:

$$X=x/x_0>0,\qquad Y=y/y_0>0,\qquad Y=A X^b,\quad A>0.$$

Así los logaritmos se aplican a razones, no a cantidades con unidades.
Las referencias se mantienen fijas durante el análisis. Cambiarlas de
forma coherente modifica el intercepto, pero no la elasticidad $b$.

### Paso 1. Transportar la función

Para $f(X)=A X^b$ y $\alpha=\exp$:

$$F(t)=\log(f(e^t))=\log A+bt.$$

El modelo es una recta en coordenadas $t=\log X$, $v=\log Y$.
La derivada transportada vale $D_\alpha f=e^b$; **su logaritmo**, $b$,
es la elasticidad. No debe confundirse $D_\alpha f$ con la derivada
ordinaria $f'$ ni decir que la derivada Alpha es directamente $b$.

Usamos la dirección $F=\alpha^{-1}\circ f\circ\alpha$, derivada de
la Definición 4.1.1 y contrastada con el Ejemplo 5.1.2. El pasaje entre
ambos contiene erratas locales de composición: véase el alcance de las
fuentes al final. Para este caso, la derivación escrita arriba es directa.

In [4]:
A = 3.0
b = 1.5
X = [0.5, 1.0, 2.0, 4.0, 8.0]
Y = [A * valor ** b for valor in X]
t = [math.log(valor) for valor in X]
v = [math.log(valor) for valor in Y]

filas_potencia = [
    [f"{x:.3f}", f"{y:.6f}", f"{tx:.6f}", f"{vy:.6f}"]
    for x, y, tx, vy in zip(X, Y, t, v)
    ]
tabla("De curva a recta", ["X", "Y", "log(X)", "log(Y)"], filas_potencia)
exportar_tabla("potencia-coordenadas.csv", ["X", "Y", "log_X", "log_Y"], filas_potencia)

for tx, vy in zip(t, v):
    assert math.isclose(vy, math.log(A) + b * tx, abs_tol=1e-12)

De curva a recta
X     | Y         | log(X)    | log(Y)  
------+-----------+-----------+---------
0.500 | 1.060660  | -0.693147 | 0.058892
1.000 | 3.000000  | 0.000000  | 1.098612
2.000 | 8.485281  | 0.693147  | 2.138333
4.000 | 24.000000 | 1.386294  | 3.178054
8.000 | 67.882251 | 2.079442  | 4.217775


### Paso 2. Ajustar la recta y comprobar el regreso

La pendiente por mínimos cuadrados en logaritmos es

$$\hat b=\frac{\sum_i(t_i-\bar t)(v_i-\bar v)}
                       {\sum_i(t_i-\bar t)^2},\qquad
\hat a=\bar v-\hat b\bar t,\quad \hat A=e^{\hat a}.$$

Este es el ajuste log-log convencional. La lectura Alpha organiza las
coordenadas y la interpretación; no introduce otro estimador.
Las sumas cuestan $O(n)$, como el baseline convencional idéntico.

In [5]:
def ajustar_recta(abscisas, ordenadas):
    """Ajusta intercepto y pendiente a datos finitos no degenerados."""
    if len(abscisas) != len(ordenadas) or len(abscisas) < 2:
        raise ValueError(
                "Se requieren listas de igual tamaño con al menos dos datos."
                )
    if not all(math.isfinite(valor) for valor in [*abscisas, *ordenadas]):
        raise ValueError("Los datos del ajuste deben ser finitos.")
    media_x = math.fsum(abscisas) / len(abscisas)
    media_y = math.fsum(ordenadas) / len(ordenadas)
    diferencias_x = [valor - media_x for valor in abscisas]
    denominador = math.fsum(valor * valor for valor in diferencias_x)
    if denominador == 0.0:
        raise ValueError("La entrada debe variar para identificar la pendiente.")
    numerador = math.fsum(
            diferencia * (valor - media_y)
            for diferencia, valor in zip(diferencias_x, ordenadas)
            )
    pendiente = numerador / denominador
    return media_y - pendiente * media_x, pendiente

In [6]:
a_estimado, b_estimado = ajustar_recta(t, v)
A_estimado = math.exp(a_estimado)
print(f"A estimado = {A_estimado:.9f}; b estimado = {b_estimado:.9f}")
print(f"Derivada Alpha = exp(b) = {math.exp(b_estimado):.9f}")

assert math.isclose(A_estimado, A, rel_tol=1e-12)
assert math.isclose(b_estimado, b, abs_tol=1e-12)

# Otra referencia de entrada: X_nueva = x / (10*x0).
t_otra_referencia = [math.log(valor / 10.0) for valor in X]
a_otro, b_otro = ajustar_recta(t_otra_referencia, v)
assert math.isclose(b_otro, b, abs_tol=1e-12)
assert math.isclose(a_otro, math.log(A) + b * math.log(10.0), abs_tol=1e-12)
print("Cambiar la referencia conserva b y desplaza el intercepto.")

A estimado = 3.000000000; b estimado = 1.500000000
Derivada Alpha = exp(b) = 4.481689070
Cambiar la referencia conserva b y desplaza el intercepto.


### Paso 3. Convertir la pendiente en una explicación

La elasticidad ordinaria de esta función es

$$\frac{Xf'(X)}{f(X)}=b.$$

Un incremento pequeño de 1% en la entrada produce aproximadamente $b$%
en la respuesta. Para un cambio finito por factor $q$, la relación exacta
es $Y'/Y=q^b$. La aproximación porcentual no es una igualdad para cualquier
cambio. Esta lectura puede guiar una hipótesis de escalamiento o una
calibración; los datos y el mecanismo deben justificar su aplicación.

In [7]:
factor_cambio = 1.01
cambio_exacto = factor_cambio ** b - 1.0
cambio_lineal = b * (factor_cambio - 1.0)
elasticidad_secante = math.log(factor_cambio ** b) / math.log(factor_cambio)

tabla(
        "Entrada +1% con b = 1.5",
        ["Cantidad", "Valor"],
        [
            ["Respuesta: cambio exacto", f"{100 * cambio_exacto:.6f}%"],
            ["Aproximación local", f"{100 * cambio_lineal:.6f}%"],
            ["Razón logarítmica exacta", f"{elasticidad_secante:.9f}"],
            ],
        )
assert math.isclose(elasticidad_secante, b, rel_tol=1e-12)

Entrada +1% con b = 1.5
Cantidad                 | Valor      
-------------------------+------------
Respuesta: cambio exacto | 1.503744%  
Aproximación local       | 1.500000%  
Razón logarítmica exacta | 1.500000000


### Paso 4. Añadir errores exige una decisión de modelo

Con error **multiplicativo**,

$$Y_i=A X_i^b e^{\varepsilon_i}
\quad\Longrightarrow\quad
\log Y_i=\log A+b\log X_i+\varepsilon_i.$$

Para interpretar log-OLS como estimación bajo ruido normal, se asumen
errores condicionales centrados, normales y con varianza constante en la
escala logarítmica, además de las condiciones del diseño. La identidad
algebraica no impone esos supuestos. En la siguiente demostración usamos
errores **elegidos y fijos**, sin afirmar que sean una muestra normal.

In [8]:
t_demo = [-2.0, -1.0, 0.0, 1.0, 2.0]
errores_elegidos = [0.15, -0.10, 0.05, 0.20, -0.05]
X_demo = [math.exp(valor) for valor in t_demo]
Y_demo = [
    A * x ** b * math.exp(error)
    for x, error in zip(X_demo, errores_elegidos)
    ]
v_demo = [math.log(valor) for valor in Y_demo]
a_demo, b_demo = ajustar_recta(t_demo, v_demo)

filas_ajuste = [
    [f"{x:.4f}", f"{y:.5f}", f"{error:+.2f}", f"{math.exp(a_demo) * x ** b_demo:.5f}"]
    for x, y, error in zip(X_demo, Y_demo, errores_elegidos)
    ]
tabla(
        "Demostración con errores fijos",
        ["X", "Y observado", "Error log", "Curva ajustada"],
        filas_ajuste,
        )
exportar_tabla(
        "potencia-ajuste.csv",
        ["X", "Y", "error_log", "curva_ajustada"],
        filas_ajuste,
        )
print(f"A ajustado = {math.exp(a_demo):.6f}; b ajustado = {b_demo:.6f}")
assert math.isclose(a_demo, math.log(A) + 0.05, abs_tol=1e-12)
assert math.isclose(b_demo, 1.49, abs_tol=1e-12)

Demostración con errores fijos
X      | Y observado | Error log | Curva ajustada
-------+-------------+-----------+---------------
0.1353 | 0.17353     | +0.15     | 0.16019       
0.3679 | 0.60569     | -0.10     | 0.71078       
1.0000 | 3.15381     | +0.05     | 3.15381       
2.7183 | 16.42184    | +0.20     | 13.99377      
7.3891 | 57.31786    | -0.05     | 62.09170      
A ajustado = 3.153813; b ajustado = 1.490000


### Límite 1. El criterio de error no se conserva al tomar logaritmos

Log-OLS minimiza $\sum_i[\log Y_i-a-b\log X_i]^2$.
El ajuste en la escala original minimiza $\sum_i[Y_i-A X_i^b]^2$.
Son preguntas diferentes.

Incluso fijando $b$ y observando sólo $X=1$, para $Y=[1,2,9]$ el ajuste
original es la media aritmética; el ajuste logarítmico vuelve como media
geométrica. Cada uno minimiza su propio criterio. No se puede declarar
mejor uno comparando errores medidos en escalas distintas.

In [9]:
observaciones = [1.0, 2.0, 9.0]
ajuste_original = math.fsum(observaciones) / len(observaciones)
media_log = math.fsum(math.log(y) for y in observaciones) / len(observaciones)
ajuste_log = math.exp(media_log)

filas_criterio = []
for nombre, ajuste in [("LS original", ajuste_original), ("LS log", ajuste_log)]:
    sse_original = math.fsum((y - ajuste) ** 2 for y in observaciones)
    sse_log = math.fsum((math.log(y) - math.log(ajuste)) ** 2 for y in observaciones)
    filas_criterio.append(
            [
                nombre,
                f"{ajuste:.6f}",
                f"{sse_original:.6f}",
                f"{sse_log:.6f}",
                ]
            )

tabla(
        "Mismos datos, distinto objetivo",
        ["Ajuste", "Predicción", "SSE original", "SSE log"],
        filas_criterio,
        )
exportar_tabla(
        "potencia-criterios.csv",
        ["ajuste", "prediccion", "sse_original", "sse_log"],
        filas_criterio,
        )
assert ajuste_original == 4.0
assert math.isclose(ajuste_log, 18.0 ** (1.0 / 3.0), rel_tol=1e-12)

Mismos datos, distinto objetivo
Ajuste      | Predicción | SSE original | SSE log 
------------+------------+--------------+---------
LS original | 4.000000   | 38.000000    | 3.059873
LS log      | 2.620741   | 43.707063    | 2.523499


### Límite 2. Volver con `exp` no da automáticamente la media

Si $\varepsilon\mid X\sim N(0,\sigma^2)$, entonces

$$\operatorname{Mediana}(Y\mid X)=A X^b,\qquad
E[Y\mid X]=A X^b e^{\sigma^2/2}.$$

La media y la mediana coinciden sólo en el caso sin varianza.
El factor escrito es válido bajo ese modelo lognormal; estimar sus
parámetros añade incertidumbre. No es una corrección universal para
cualquier distribución ni para varianza que cambie con $X$.

In [10]:
filas_media = [
    [f"{sigma:.1f}", f"{A:.6f}", f"{A * math.exp(sigma ** 2 / 2):.6f}"]
    for sigma in [0.0, 0.3, 0.6, 1.0]
    ]
tabla("En X = 1, A = 3: modelo lognormal", ["sigma", "Mediana", "Media"], filas_media)
exportar_tabla("potencia-media-mediana.csv", ["sigma", "mediana", "media"], filas_media)
assert math.exp(0.6 ** 2 / 2) > 1.0

En X = 1, A = 3: modelo lognormal
sigma | Mediana  | Media   
------+----------+---------
0.0   | 3.000000 | 3.000000
0.3   | 3.000000 | 3.138084
0.6   | 3.000000 | 3.591652
1.0   | 3.000000 | 4.946164


### Límite 3. El error aditivo puede salir del dominio

Si el modelo fuera $Y=A X^b+\eta$, no se obtiene el modelo multiplicativo
por cambiar su escritura. Además, valores observados cero o negativos
impiden usar estos logaritmos. La siguiente comprobación representa
$X=1$, $A=3$ y errores aditivos $[-4,0,4]$.

No sumamos automáticamente una constante para «arreglar» el dominio:
esa decisión cambia la función y su interpretación.

In [11]:
respuestas_aditivas = [A + error for error in [-4.0, 0.0, 4.0]]
print("Respuestas con error aditivo:", respuestas_aditivas)

try:
    [log_positivo(y) for y in respuestas_aditivas]
except ValueError as error:
    print("Dominio rechazado correctamente:", error)
else:
    raise AssertionError("Debía rechazarse la respuesta negativa.")

Respuestas con error aditivo: [-1.0, 3.0, 7.0]
Dominio rechazado correctamente: La cantidad debe ser positiva y finita.


### Ejercicio

Una curva es $Y=2X^{0.5}$. ¿Qué pasa al duplicar $X$? ¿Cuánto vale
su elasticidad? ¿Y su derivada Alpha para $\alpha=\exp$?

**Pista:** separa el factor exacto $2^{0.5}$, el exponente $0.5$ y
$e^{0.5}$. Después explica por qué ninguno necesita justificar un modelo
de ruido para esta pregunta determinista.

In [12]:
b_ejercicio = 0.5
print(f"Factor al duplicar X: {2.0 ** b_ejercicio:.9f}")
print(f"Elasticidad: {b_ejercicio}")
print(f"Derivada Alpha: {math.exp(b_ejercicio):.9f}")
assert math.isclose(2.0 ** b_ejercicio, math.sqrt(2.0), rel_tol=1e-12)

Factor al duplicar X: 1.414213562
Elasticidad: 0.5
Derivada Alpha: 1.648721271


## 2. Qué quedó comprobado y qué necesita más evidencia

El cambio a coordenadas logarítmicas convierte una potencia positiva
en una recta. Su pendiente es la elasticidad constante: permite explicar
cambios relativos y reconocer hipótesis de escalamiento o calibración.
La derivada Alpha es $e^b$, y su logaritmo es $b$.

El ajuste es la regresión log-log convencional. La lectura Alpha organiza
las coordenadas y su interpretación, sin añadir un estimador nuevo ni una
ventaja exclusiva. La demostración no prueba una ley empírica: los datos
son elegidos. Transformar la respuesta modifica el criterio de error y
exige comprobar el modelo de ruido y qué predicción queremos comunicar.

**Extensión propuesta:** elegir datos públicos y justificar el mecanismo,
las unidades de referencia y el error antes de validar ajuste e inferencia.

## Fuentes y alcance

- Georgiev y Aydin, *Alpha Calculus*, 1.ª ed., 2026,
  [DOI 10.1201/9781003750734](https://doi.org/10.1201/9781003750734).
  Pasajes cotejados: pp. **1–2** (generador), **128** (Definición 4.1.1),
  **169–170** (ecuaciones 5.1–5.2 y Ejemplo 5.1.2). En este PDF
  corresponden a 10–11, 137 y 178–179. En pp. 169–170 hay omisiones
  locales e inversión de composición documentadas por el lector, aún
  pendientes de confirmación editorial. No usamos la igualdad invertida:
  aquí la conjugación y la derivada se derivan directamente.
- [NIST/SEMATECH, 2.3.6.1: calibración](https://www.itl.nist.gov/div898/handbook/mpc/section3/mpc361.htm),
  «Power model treated as a linear model»: error multiplicativo y
  linealización. Las referencias adimensionales y los datos son nuestra
  exposición.
- [NIST/SEMATECH, 1.3.6.6.9: distribución lognormal](https://www.itl.nist.gov/div898/handbook/eda/section3/eda3669.htm),
  parametrización y «Common Statistics»: media y mediana. La fórmula
  condicional se aplica bajo los supuestos declarados.
- [Python: `math`](https://docs.python.org/3.14/library/math.html),
  `log`, `exp`, `fsum` e `isclose`: operaciones de implementación.

Se consultaron esos apartados durante la preparación. Las derivaciones,
los ejemplos y el código son elaboración didáctica del laboratorio.
No se reivindica prioridad científica ni se distribuye el libro. Tampoco
se certifica la totalidad de sus resultados.

In [13]:
resumen = {
    "estado": "passed",
    "aplicacion": "potencias_y_elasticidad",
    "python": platform.python_version(),
    "A_sin_ruido": A_estimado,
    "b_sin_ruido": b_estimado,
    "A_errores_fijos": math.exp(a_demo),
    "b_errores_fijos": b_demo,
    "criterio_original": ajuste_original,
    "criterio_log": ajuste_log,
    "dominio_negativo_rechazado": True,
    "modelo_empirico_validado": False,
    }
(RESULTADOS / "comprobaciones.json").write_text(
        json.dumps(resumen, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        )
print("Comprobaciones aprobadas. CSV y resumen en:", RESULTADOS)

Comprobaciones aprobadas. CSV y resumen en: resultados/potencias_y_elasticidad
